# SRT-M15：无需服务器的 GPU 分批训练（原生自主模型）

> 此 Notebook 在 **Google Colab 提供 GPU 时** 才训练。训练代码来自你的 GitHub `main/SRT-M15`，断点保存在 **你自己的 Google Drive**。不需要购买服务器，也不会向第三方上传你的 Google Drive 内容。

**需要自己操作两处授权：** Colab 菜单「运行时 → 更改运行时类型 → GPU」，随后执行第 1 格时允许挂载 Drive。免费 GPU 不保证随时可用；每次训练会受到 Colab 会话时长及额度限制。

**模型来源披露：** 本 Notebook 只训练原生、随机初始化的 SRT-M15 小模型（**不借用 GPT/Claude/Qwen 权重**）。10 万条新增样本主要是模板合成数据，不能保证学会普遍中文、英文。

**推荐使用方式：** 按顺序执行全部单元格。训练一次默认最多 **10 批 × 320 优化步**；每批完成后立即把 `latest.pt`、`best.pt`、`history.json` 写入 Drive。中断后再次运行会自动从保存的 `latest.pt` 继续，最多只损失当前尚未完成的一批。

In [ ]:
# 第 1 步：挂载你自己的 Google Drive；这些数据不进入 GitHub 仓库。
from google.colab import drive
from pathlib import Path
import os, shutil, subprocess, hashlib, json

drive.mount("/content/drive")
PERSISTENT_DIR = Path("/content/drive/MyDrive/SRT-M15/original_110k_gpu")
PERSISTENT_DIR.mkdir(parents=True, exist_ok=True)
print("持久化检查点目录:", PERSISTENT_DIR)


In [ ]:
# 第 2 步：获取 GitHub 仓库最新代码。首次运行时自动 clone。
from pathlib import Path
import os, subprocess
ROOT = Path("/content/srt_m15_colab_src")
if not (ROOT / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main",
                    "https://github.com/mkusidodohsvebdks888/x.git", str(ROOT)], check=True)
else:
    subprocess.run(["git", "-C", str(ROOT), "fetch", "--depth", "1", "origin", "main"], check=True)
    subprocess.run(["git", "-C", str(ROOT), "reset", "--hard", "origin/main"], check=True)
PROJECT = ROOT / "SRT-M15"
os.chdir(PROJECT)
REVISION = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip()
print("GitHub 版本:", REVISION)


In [ ]:
# 第 3 步：检查 GPU；没有 GPU 时明确停止，不会假装训练成功。
import torch
if not torch.cuda.is_available():
    raise RuntimeError("当前 Colab 没有 GPU。请在「运行时 > 更改运行时类型」选 GPU 再重新执行。")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
print("PyTorch:", torch.__version__)


In [ ]:
# 第 4 步：离线生成原有 10k + 新增 100k 双语合成样本，并验证 SHA256。
from pathlib import Path
import hashlib, json, subprocess, sys
DATA = PROJECT / "examples/bilingual_synthetic_v4_110000.jsonl"
MANIFEST = PROJECT / "examples/bilingual_synthetic_v4_manifest.json"
subprocess.run([sys.executable, "generate_bilingual_110k.py",
                "--output", str(DATA), "--manifest", str(MANIFEST)], check=True)
report = json.loads(MANIFEST.read_text(encoding="utf-8"))
assert report["total_records"] == 110000
assert report["added_records"] == 100000
assert hashlib.sha256(DATA.read_bytes()).hexdigest() == report["data_sha256"]
print("中文+英文:", report["by_language"])
print("训练/验证:", report["train_records"], report["dev_records"])
print("SHA256:", report["data_sha256"])


In [ ]:
# 第 5 步：设置本次可控批次，校验 Drive 中的模型是否属于相同数据/超参数。
from pathlib import Path
import json, shutil

# 每轮 Colab 最多追加多少批？可调整 1..10，不影响单批训练参数。
BATCHES_THIS_SESSION = 10
STEPS_PER_BATCH = 320    # 已保存检查点的参数：续训时必须保持一致
BATCH_SIZE = 32
CONTEXT = 256
MAX_EVAL_PAIRS = 512
SEED = 42

if not 1 <= BATCHES_THIS_SESSION <= 10:
    raise ValueError("每次运行批数必须在 1 到 10 之间")
meta = {
    "schema": 1, "dataset_sha256": report["data_sha256"], "seed": SEED,
    "steps_per_batch": STEPS_PER_BATCH, "batch_size": BATCH_SIZE,
    "context": CONTEXT, "max_eval_pairs": MAX_EVAL_PAIRS,
    "sampling": "cycle", "checkpoint_type": "original_srt_m15_not_lora",
}
meta_path = PERSISTENT_DIR / "training_spec.json"
if meta_path.exists():
    previous = json.loads(meta_path.read_text(encoding="utf-8"))
    if {k:previous.get(k) for k in meta} != meta:
        raise ValueError("Drive 中已有不同的数据或超参数检查点，请使用新目录，不要覆盖旧版模型。")
else:
    if (PERSISTENT_DIR / "latest.pt").exists():
        raise ValueError("发现没有 training_spec.json 的旧检查点，请先备份并核对后再恢复。")
    meta_path.write_text(json.dumps({**meta, "initial_git_revision": REVISION},
                                    ensure_ascii=False, indent=2)+"\n", encoding="utf-8")
print("已验证检查点兼容性。", "历史训练:", (PERSISTENT_DIR/"latest.pt").exists())


In [ ]:
# 第 6 步：每次训练一批，成功后立即原子写入 Drive；中断可跨 Colab 会话续训。
import os, shutil, subprocess, sys, json, time
from pathlib import Path

LOCAL = PROJECT / "checkpoints/colab_stages"
LOCAL.mkdir(parents=True, exist_ok=True)

def persist(src: Path, name: str):
    target = PERSISTENT_DIR / name
    temporary = target.with_name(target.name+".uploading")
    shutil.copyfile(src, temporary)
    os.replace(temporary, target)

if (PERSISTENT_DIR / "latest.pt").is_file():
    for name in ("latest.pt", "best.pt", "history.json"):
        src = PERSISTENT_DIR / name
        if src.is_file():
            shutil.copyfile(src, LOCAL / name)

for i in range(BATCHES_THIS_SESSION):
    cmd = [sys.executable, "train_bilingual_stages.py",
           "--data", str(DATA), "--allow-demo-data",
           "--sampling", "cycle", "--device", "cuda",
           "--context", str(CONTEXT), "--batch-size", str(BATCH_SIZE),
           "--max-eval-pairs", str(MAX_EVAL_PAIRS),
           "--stages", "1", "--steps-per-stage", str(STEPS_PER_BATCH),
           "--seed", str(SEED), "--output", str(LOCAL)]
    if (LOCAL / "latest.pt").exists():
        cmd += ["--resume", str(LOCAL / "latest.pt")]
    print(f"\n第 {i+1}/{BATCHES_THIS_SESSION} 批开始。成功后马上备份到 Google Drive。", flush=True)
    subprocess.run(cmd, cwd=PROJECT, check=True)
    for name in ("latest.pt", "best.pt", "history.json"):
        src = LOCAL / name
        if src.is_file():
            persist(src, name)
    latest_report = json.loads((LOCAL/"history.json").read_text(encoding="utf-8"))
    print("已保存，累计:", latest_report["stage"], "批 /", latest_report["total_steps"], "步", flush=True)

print("本次分批运行完成。最新权重已经保存在 Google Drive。")


In [ ]:
# 第 7 步：查看已保存的中文、英文验证指标与训练进度。
import json
from pathlib import Path
report_file = PERSISTENT_DIR / "history.json"
if report_file.exists():
    info = json.loads(report_file.read_text(encoding="utf-8"))
    print("已完成批次:", info["stage"], "累计步数:", info["total_steps"])
    print("当前最佳批次:", info["best_stage"])
    print("各语种已覆盖训练窗口:", info.get("cumulative_unique_window_coverage"))
    if info["history"]:
        last = info["history"][-1]
        for lang in ("zh", "en"):
            print(lang, last["dev"][lang])
else:
    print("还没有成功保存批次检查点。")


## 训练结束后如何继续？

1. Colab 断线或免费 GPU 时段用完，不需要重新开始整个模型。下一次打开相同 Notebook，挂载**同一个** Google Drive 目录，从第一格开始运行，即可读取已有检查点并继续。
2. 目前的 GPU 训练仍针对 **原生 SRT-M15 小型网络**。数据量大不等于模型语言能力强；有必要通过独立真实对话评估并研究扩大参数与改进 Tokenizer。
3. `latest.pt` 和 `best.pt` 仅保存在你的 Drive。**不会自动向 GitHub 上传你的 Drive、模型权重或私密文件。** 如果需要公开发布模型，应先检查数据来源、许可和隐私。
4. 如 GPU 可用但显存不足，可以减小 `BATCH_SIZE` 或 `CONTEXT`，但必须改用**新的 Drive 目录重新开始训练**；不能拿旧 checkpoint 强行继续不同配置。
5. 由于每批单独进程启动，大量文本/验证构建会重复占用 CPU 时间。这是优先保证断线可恢复的研究方案，并非高吞吐生产训练。

**实用中英对话替代路线：** 项目已有 `bilingual_runtime.py` 和 `finetune_bilingual.py`，可在公开预训练模型上 LoRA 微调；这是另一条路线，不是原生 SRT-M15 的参数训练。
